# 📱 Member 2 — Vision Transformer (ViT) + Image Augmentation
## Second-Hand Mobile Price Analyzer — Egypt (Dubizzle)

---

### 🎯 What this notebook covers
| Step | Description |
|------|-------------|
| 1 | Setup & library installation |
| 2 | Load & explore the labeled dataset (5,358 listings) |
| 3 | Download mobile images from Dubizzle URLs |
| 4 | Image augmentation pipeline |
| 5 | Fine-tune a pre-trained ViT (`google/vit-base-patch16-224`) |
| 6 | Evaluation — accuracy, F1, confusion matrix, ROC |
| 7 | Compare ViT vs CNN performance |
| 8 | Extract & **save image feature vectors** for Member 5 (Multimodal) |

### 📂 Output files saved for the team
```
outputs/
├── vit_image_features.npy          ← image embeddings (N × 768) for Member 5
├── vit_image_features_meta.csv     ← listing_url + label aligned with features
├── vit_best_model/                 ← saved ViT model weights
├── vit_metrics.json                ← all evaluation metrics
└── vit_augmented_samples.png       ← augmentation visualisation
```

### 📊 Dataset
- **Source:** Dubizzle Egypt — scraped by Member 1
- **Rows:** 5,358 listings
- **Labels:** `Fair` (4,440) · `Overpriced` (918)
- **Columns:** `listing_id`, `price_egp`, `description`, `clean_description`, `image_url`, `listing_url`, `brand`, `label`, `label_int`

---
## Cell 1 — Install Dependencies

In [ ]:
# ── Install all required packages ─────────────────────────────────────────
import subprocess, sys

packages = [
    "torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118",
    "transformers>=4.38.0",
    "timm",
    "datasets",
    "scikit-learn",
    "pandas",
    "numpy",
    "matplotlib",
    "seaborn",
    "Pillow",
    "requests",
    "tqdm",
    "albumentations",
    "opencv-python-headless",
    "accelerate",
]

for pkg in packages:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + pkg.split(), check=False)

print("✅ All packages installed.")

---
## Cell 2 — Imports & Global Config

In [ ]:
# ── Standard library ──────────────────────────────────────────────────────
import os, json, hashlib, shutil, warnings, time, random
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

# ── Data ─────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd

# ── Vision ────────────────────────────────────────────────────────────────
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from PIL import Image
import requests
import albumentations as A
from albumentations.pytorch import ToTensorV2
import cv2

# ── HuggingFace Transformers ──────────────────────────────────────────────
from transformers import (
    ViTForImageClassification,
    ViTFeatureExtractor,
    ViTModel,
    AutoFeatureExtractor,
    TrainingArguments,
    Trainer,
)

# ── Metrics ───────────────────────────────────────────────────────────────
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report,
    confusion_matrix, roc_auc_score, roc_curve,
    precision_score, recall_score,
)
from sklearn.model_selection import train_test_split

# ── Plotting ──────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from tqdm.notebook import tqdm

warnings.filterwarnings("ignore")

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# GLOBAL CONFIG  — adjust paths here only
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
CFG = {
    # ── Paths ──────────────────────────────────────────────────────────────
    "labeled_csv"    : "labeled_dataset.csv",     # from Member 1 / Member 4
    "img_dir"        : "downloaded_images",        # where raw images are saved
    "proc_dir"       : "preprocessed_images",      # 224×224 resized images
    "output_dir"     : "outputs",                  # all saved artefacts

    # ── Model ──────────────────────────────────────────────────────────────
    "vit_model_name" : "google/vit-base-patch16-224",  # HuggingFace model id
    "num_labels"     : 2,
    "label2id"       : {"Fair": 0, "Overpriced": 1},
    "id2label"       : {0: "Fair",  1: "Overpriced"},

    # ── Training ───────────────────────────────────────────────────────────
    "img_size"       : 224,
    "batch_size"     : 16,
    "epochs"         : 5,
    "lr"             : 2e-5,
    "weight_decay"   : 0.01,
    "val_split"      : 0.15,
    "test_split"     : 0.15,
    "seed"           : 42,

    # ── Download ───────────────────────────────────────────────────────────
    "max_workers"    : 12,
    "timeout"        : 15,
    "max_images"     : None,   # None = download all; set e.g. 500 for quick test
}

# Create output directories
for d in [CFG["img_dir"], CFG["proc_dir"], CFG["output_dir"], "outputs/vit_best_model"]:
    Path(d).mkdir(parents=True, exist_ok=True)

# Reproducibility
torch.manual_seed(CFG["seed"])
np.random.seed(CFG["seed"])
random.seed(CFG["seed"])

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"🖥️  Device : {DEVICE}")
print(f"🔥  CUDA   : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   GPU    : {torch.cuda.get_device_name(0)}")
print(f"📁  Outputs: {CFG['output_dir']}/")

---
## Cell 3 — Load & Explore the Labeled Dataset

In [ ]:
# ── Load dataset ──────────────────────────────────────────────────────────
df = pd.read_csv(CFG["labeled_csv"])
print(f"📊 Dataset shape: {df.shape}")
print(f"   Columns: {list(df.columns)}")
df.head()

In [ ]:
# ── EDA: Label distribution ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Dataset Exploration — Dubizzle Egypt Mobile Listings", fontsize=14, fontweight="bold")

# Label counts
label_counts = df["label"].value_counts()
colors = ["#2ecc71", "#e74c3c"]
axes[0].bar(label_counts.index, label_counts.values, color=colors, edgecolor="white", linewidth=1.5)
axes[0].set_title("Label Distribution", fontweight="bold")
axes[0].set_ylabel("Count")
for i, v in enumerate(label_counts.values):
    axes[0].text(i, v + 30, str(v), ha="center", fontweight="bold")

# Price distribution by label
for label, color in zip(["Fair", "Overpriced"], colors):
    subset = df[df["label"] == label]["price_egp"]
    axes[1].hist(subset, bins=40, alpha=0.6, color=color, label=label, edgecolor="white")
axes[1].set_title("Price Distribution by Label", fontweight="bold")
axes[1].set_xlabel("Price (EGP)")
axes[1].set_ylabel("Count")
axes[1].legend()
axes[1].set_xlim(0, 80000)

# Brand distribution
top_brands = df["brand"].value_counts().head(8)
axes[2].barh(top_brands.index[::-1], top_brands.values[::-1], color="#3498db", edgecolor="white")
axes[2].set_title("Top Brands", fontweight="bold")
axes[2].set_xlabel("Count")

plt.tight_layout()
plt.savefig(f"{CFG['output_dir']}/eda_overview.png", dpi=150, bbox_inches="tight")
plt.show()

print("\n📈 Price statistics by label:")
print(df.groupby("label")["price_egp"].describe().round(0))

In [ ]:
# ── Check for missing image URLs ──────────────────────────────────────────
df_clean = df.dropna(subset=["image_url"]).copy()
df_clean = df_clean[df_clean["image_url"].str.startswith("http")].reset_index(drop=True)

# Apply optional cap for quick testing
if CFG["max_images"] is not None:
    df_clean = df_clean.sample(CFG["max_images"], random_state=CFG["seed"]).reset_index(drop=True)

print(f"✅ Clean rows with valid image URLs: {len(df_clean):,}")
print(f"   Label breakdown:")
print(df_clean["label"].value_counts())

---
## Cell 4 — Download Images from Dubizzle

In [ ]:
# ── Image downloader (parallel, crash-safe) ───────────────────────────────

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/124.0.0.0 Safari/537.36",
    "Accept": "image/webp,image/apng,image/*,*/*;q=0.8",
    "Referer": "https://www.dubizzle.com.eg/",
}


def slugify_url(url: str) -> str:
    """Deterministic, filesystem-safe filename derived from the URL."""
    h = hashlib.sha1(url.encode()).hexdigest()[:12]
    ext = Path(url.split("?")[0]).suffix.lower()
    if ext not in {".jpg", ".jpeg", ".png", ".webp"}:
        ext = ".jpg"
    return f"{h}{ext}"


def download_one(args):
    """Download a single image. Returns (local_path, success)."""
    url, dest_path = args
    if dest_path.exists():
        return dest_path, True   # already cached
    try:
        resp = requests.get(url, headers=HEADERS, timeout=CFG["timeout"], stream=True)
        resp.raise_for_status()
        ct = resp.headers.get("content-type", "").lower()
        if "image" not in ct and "octet-stream" not in ct:
            return dest_path, False
        with open(dest_path, "wb") as f:
            for chunk in resp.iter_content(8192):
                f.write(chunk)
        return dest_path, True
    except Exception:
        return dest_path, False


def download_all_images(df: pd.DataFrame, img_dir: str) -> pd.DataFrame:
    """Download all images and return df with 'local_path' column."""
    img_dir = Path(img_dir)
    img_dir.mkdir(parents=True, exist_ok=True)

    tasks = [
        (row.image_url, img_dir / slugify_url(row.image_url))
        for row in df.itertuples()
    ]

    results = {}
    with ThreadPoolExecutor(max_workers=CFG["max_workers"]) as ex:
        futures = {ex.submit(download_one, t): i for i, t in enumerate(tasks)}
        for fut in tqdm(as_completed(futures), total=len(futures), desc="Downloading images"):
            idx = futures[fut]
            path, ok = fut.result()
            results[idx] = (str(path), ok)

    df = df.copy()
    df["local_path"] = [results[i][0] for i in range(len(df))]
    df["download_ok"] = [results[i][1] for i in range(len(df))]

    ok = df["download_ok"].sum()
    print(f"\n✅ Downloaded {ok:,} / {len(df):,} images  "
          f"(failed: {len(df)-ok:,})")
    return df


df_imgs = download_all_images(df_clean, CFG["img_dir"])
df_imgs = df_imgs[df_imgs["download_ok"]].reset_index(drop=True)
print(f"📦 Usable images: {len(df_imgs):,}")

---
## Cell 5 — Image Augmentation Pipeline

In [ ]:
# ── Augmentation transforms (Albumentations) ──────────────────────────────

IMG_MEAN = [0.5, 0.5, 0.5]   # ViT uses 0.5 normalisation
IMG_STD  = [0.5, 0.5, 0.5]
SIZE     = CFG["img_size"]

# ── Training augmentations (aggressive, diverse) ──────────────────────────
train_aug = A.Compose([
    A.Resize(SIZE, SIZE),

    # ── Geometric ─────────────────────────────────────────────────────────
    A.HorizontalFlip(p=0.5),
    A.Rotate(limit=15, p=0.4),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=10, p=0.4),
    A.RandomCrop(height=200, width=200, p=0.3),
    A.Resize(SIZE, SIZE),   # restore size after crop

    # ── Colour / lighting ─────────────────────────────────────────────────
    A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.5),
    A.HueSaturationValue(hue_shift_limit=20, sat_shift_limit=30, val_shift_limit=20, p=0.4),
    A.RandomGamma(p=0.3),
    A.CLAHE(p=0.2),

    # ── Noise / blur ──────────────────────────────────────────────────────
    A.GaussNoise(var_limit=(10, 50), p=0.3),
    A.GaussianBlur(blur_limit=(3, 5), p=0.2),
    A.MotionBlur(blur_limit=5, p=0.15),

    # ── Dropout / occlusion ───────────────────────────────────────────────
    A.CoarseDropout(max_holes=8, max_height=20, max_width=20, p=0.3),

    # ── Normalise & convert to tensor ─────────────────────────────────────
    A.Normalize(mean=IMG_MEAN, std=IMG_STD),
    ToTensorV2(),
])

# ── Validation / test augmentations (resize + normalize only) ─────────────
val_aug = A.Compose([
    A.Resize(SIZE, SIZE),
    A.Normalize(mean=IMG_MEAN, std=IMG_STD),
    ToTensorV2(),
])

print("✅ Augmentation pipelines defined.")
print(f"   Train transforms : {len(train_aug.transforms)} steps")
print(f"   Val   transforms : {len(val_aug.transforms)} steps")

In [ ]:
# ── Visualise augmentation on sample images ────────────────────────────────

def show_augmentation_grid(df: pd.DataFrame, n_images: int = 4, n_aug: int = 5):
    """Show n_images originals and n_aug augmented versions each."""
    samples = df.sample(n_images, random_state=CFG["seed"])

    fig, axes = plt.subplots(n_images, n_aug + 1, figsize=(3*(n_aug+1), 3*n_images))
    fig.suptitle("Image Augmentation Visualisation", fontsize=14, fontweight="bold")

    # Augmentation used ONLY for display (no ToTensorV2 so PIL stays)
    display_aug = A.Compose([
        A.Resize(SIZE, SIZE),
        A.HorizontalFlip(p=0.5),
        A.Rotate(limit=15, p=0.7),
        A.RandomBrightnessContrast(p=0.7),
        A.GaussNoise(var_limit=(10, 50), p=0.5),
        A.CoarseDropout(max_holes=6, max_height=20, max_width=20, p=0.5),
        A.HueSaturationValue(p=0.5),
    ])

    for row_idx, (_, row) in enumerate(samples.iterrows()):
        try:
            img = np.array(Image.open(row["local_path"]).convert("RGB").resize((SIZE, SIZE)))
        except Exception:
            continue

        # Original
        ax = axes[row_idx][0]
        ax.imshow(img)
        ax.set_title(f"Original\n{row['label']}", fontsize=8, fontweight="bold")
        ax.axis("off")

        # Augmented
        for col_idx in range(1, n_aug + 1):
            aug_img = display_aug(image=img.copy())["image"]
            axes[row_idx][col_idx].imshow(aug_img)
            axes[row_idx][col_idx].set_title(f"Aug {col_idx}", fontsize=8)
            axes[row_idx][col_idx].axis("off")

    plt.tight_layout()
    save_path = f"{CFG['output_dir']}/vit_augmented_samples.png"
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"💾 Saved → {save_path}")


show_augmentation_grid(df_imgs)

---
## Cell 6 — PyTorch Dataset & DataLoaders

In [ ]:
# ── Train / Val / Test split ──────────────────────────────────────────────
label2id = CFG["label2id"]

# Stratified split to preserve class balance
df_train, df_temp = train_test_split(
    df_imgs, test_size=CFG["val_split"] + CFG["test_split"],
    stratify=df_imgs["label"], random_state=CFG["seed"]
)
df_val, df_test = train_test_split(
    df_temp, test_size=0.5,
    stratify=df_temp["label"], random_state=CFG["seed"]
)

print(f"📦 Split sizes:")
print(f"   Train : {len(df_train):,}  ({df_train['label'].value_counts().to_dict()})")
print(f"   Val   : {len(df_val):,}  ({df_val['label'].value_counts().to_dict()})")
print(f"   Test  : {len(df_test):,}  ({df_test['label'].value_counts().to_dict()})")

In [ ]:
# ── Custom Dataset ────────────────────────────────────────────────────────

class MobileImageDataset(Dataset):
    """
    PyTorch Dataset for mobile phone listing images.
    Loads images from local disk and applies Albumentations transforms.
    """

    def __init__(self, df: pd.DataFrame, transform=None, label2id: dict = None):
        self.df        = df.reset_index(drop=True)
        self.transform = transform
        self.label2id  = label2id or {"Fair": 0, "Overpriced": 1}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # ── Load image ────────────────────────────────────────────────────
        try:
            img = Image.open(row["local_path"]).convert("RGB")
            img = np.array(img)
        except Exception:
            # Fallback: grey image so training doesn't crash on a bad file
            img = np.zeros((CFG["img_size"], CFG["img_size"], 3), dtype=np.uint8)

        # ── Apply augmentation ────────────────────────────────────────────
        if self.transform is not None:
            augmented = self.transform(image=img)
            img = augmented["image"]   # shape: (C, H, W), torch.Tensor
        else:
            img = torch.tensor(img).permute(2, 0, 1).float() / 255.0

        label = self.label2id[row["label"]]
        return {"pixel_values": img, "labels": torch.tensor(label, dtype=torch.long)}


# ── Instantiate datasets & loaders ────────────────────────────────────────
train_ds = MobileImageDataset(df_train, transform=train_aug, label2id=label2id)
val_ds   = MobileImageDataset(df_val,   transform=val_aug,   label2id=label2id)
test_ds  = MobileImageDataset(df_test,  transform=val_aug,   label2id=label2id)

train_loader = DataLoader(train_ds, batch_size=CFG["batch_size"], shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=CFG["batch_size"], shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=CFG["batch_size"], shuffle=False, num_workers=2, pin_memory=True)

print("✅ Datasets and DataLoaders ready.")
print(f"   Batches per epoch — Train: {len(train_loader)} | Val: {len(val_loader)} | Test: {len(test_loader)}")

# Quick sanity check
batch = next(iter(train_loader))
print(f"   Batch pixel_values shape : {batch['pixel_values'].shape}")
print(f"   Batch labels shape       : {batch['labels'].shape}")

---
## Cell 7 — Load Pre-trained ViT Model

In [ ]:
# ── Load ViT for image classification ────────────────────────────────────

model = ViTForImageClassification.from_pretrained(
    CFG["vit_model_name"],
    num_labels     = CFG["num_labels"],
    id2label       = CFG["id2label"],
    label2id       = CFG["label2id"],
    ignore_mismatched_sizes = True,   # replaces pretrained head with our 2-class head
)
model = model.to(DEVICE)

# ── Count parameters ──────────────────────────────────────────────────────
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"🤖 Model : {CFG['vit_model_name']}")
print(f"   Total parameters     : {total_params:,}")
print(f"   Trainable parameters : {trainable_params:,}")
print(f"   Classification head  : {model.classifier}")

---
## Cell 8 — Training Loop

In [ ]:
# ── Training utilities ────────────────────────────────────────────────────

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=CFG["lr"],
    weight_decay=CFG["weight_decay"]
)

# Cosine LR scheduler with warm-up
total_steps   = CFG["epochs"] * len(train_loader)
warmup_steps  = int(0.1 * total_steps)
scheduler     = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=total_steps - warmup_steps, eta_min=1e-6
)

# Weighted loss to handle class imbalance (Fair: 4440, Overpriced: 918)
class_counts = np.array([4440, 918])
weights      = torch.tensor(1.0 / class_counts, dtype=torch.float32).to(DEVICE)
weights      = weights / weights.sum()
criterion    = nn.CrossEntropyLoss(weight=weights)

print(f"⚙️  Optimizer   : AdamW  lr={CFG['lr']}  wd={CFG['weight_decay']}")
print(f"   Scheduler   : CosineAnnealingLR  (warmup={warmup_steps} steps)")
print(f"   Loss        : CrossEntropyLoss  (weighted: {weights.cpu().numpy().round(3)})")
print(f"   Total steps : {total_steps}")

In [ ]:
# ── Training & validation functions ───────────────────────────────────────

def train_one_epoch(model, loader, optimizer, scheduler, criterion, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0

    for batch in tqdm(loader, desc="  Training", leave=False):
        pixel_values = batch["pixel_values"].to(device)
        labels       = batch["labels"].to(device)

        optimizer.zero_grad()
        outputs = model(pixel_values=pixel_values)
        logits  = outputs.logits

        loss = criterion(logits, labels)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

        preds = logits.argmax(dim=-1)
        correct    += (preds == labels).sum().item()
        total      += labels.size(0)
        total_loss += loss.item() * labels.size(0)

    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, all_preds, all_labels, all_probs = 0.0, [], [], []

    for batch in tqdm(loader, desc="  Evaluating", leave=False):
        pixel_values = batch["pixel_values"].to(device)
        labels       = batch["labels"].to(device)

        outputs = model(pixel_values=pixel_values)
        logits  = outputs.logits
        loss    = criterion(logits, labels)

        probs  = torch.softmax(logits, dim=-1)
        preds  = logits.argmax(dim=-1)

        total_loss  += loss.item() * labels.size(0)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs[:, 1].cpu().numpy())  # P(Overpriced)

    n   = len(all_labels)
    acc = accuracy_score(all_labels, all_preds)
    f1  = f1_score(all_labels, all_preds, average="weighted")

    return {
        "loss"   : total_loss / n,
        "acc"    : acc,
        "f1"     : f1,
        "preds"  : np.array(all_preds),
        "labels" : np.array(all_labels),
        "probs"  : np.array(all_probs),
    }

print("✅ Training functions defined.")

In [ ]:
# ── Main training loop ────────────────────────────────────────────────────

history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": [], "val_f1": []}
best_val_f1  = 0.0
best_model_path = f"{CFG['output_dir']}/vit_best_model"

print(f"🚀 Starting training for {CFG['epochs']} epochs ...\n")

for epoch in range(1, CFG["epochs"] + 1):
    t0 = time.time()

    # ── Train ──────────────────────────────────────────────────────────────
    train_loss, train_acc = train_one_epoch(
        model, train_loader, optimizer, scheduler, criterion, DEVICE
    )

    # ── Validate ───────────────────────────────────────────────────────────
    val_res = evaluate(model, val_loader, criterion, DEVICE)

    elapsed = time.time() - t0

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_res["loss"])
    history["val_acc"].append(val_res["acc"])
    history["val_f1"].append(val_res["f1"])

    # ── Save best model ────────────────────────────────────────────────────
    if val_res["f1"] > best_val_f1:
        best_val_f1 = val_res["f1"]
        model.save_pretrained(best_model_path)
        flag = "  ✅ Best"
    else:
        flag = ""

    print(
        f"Epoch {epoch:02d}/{CFG['epochs']}  "
        f"| Train loss: {train_loss:.4f}  acc: {train_acc:.4f}  "
        f"| Val loss: {val_res['loss']:.4f}  acc: {val_res['acc']:.4f}  F1: {val_res['f1']:.4f}  "
        f"| {elapsed:.0f}s"
        f"{flag}"
    )

print(f"\n🏆 Best Val F1: {best_val_f1:.4f}  → saved to {best_model_path}")

In [ ]:
# ── Training curves ───────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("ViT Training History", fontsize=14, fontweight="bold")
epochs_range = range(1, CFG["epochs"] + 1)

# Loss
axes[0].plot(epochs_range, history["train_loss"], "b-o", label="Train")
axes[0].plot(epochs_range, history["val_loss"],   "r-o", label="Val")
axes[0].set_title("Loss")
axes[0].set_xlabel("Epoch")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy
axes[1].plot(epochs_range, history["train_acc"], "b-o", label="Train")
axes[1].plot(epochs_range, history["val_acc"],   "r-o", label="Val")
axes[1].set_title("Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# F1
axes[2].plot(epochs_range, history["val_f1"], "g-o")
axes[2].set_title("Validation F1 (Weighted)")
axes[2].set_xlabel("Epoch")
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f"{CFG['output_dir']}/vit_training_curves.png", dpi=150, bbox_inches="tight")
plt.show()

---
## Cell 9 — Evaluation Metrics (Test Set)

In [ ]:
# ── Load best model and evaluate on test set ──────────────────────────────

print("🔄 Loading best saved model ...")
best_model = ViTForImageClassification.from_pretrained(
    best_model_path,
    num_labels = CFG["num_labels"],
    id2label   = CFG["id2label"],
    label2id   = CFG["label2id"],
    ignore_mismatched_sizes = True,
).to(DEVICE)

test_res = evaluate(best_model, test_loader, criterion, DEVICE)

print("\n" + "=" * 55)
print(f"  ViT Test Results")
print("=" * 55)
print(f"  Accuracy  : {test_res['acc']:.4f}")
print(f"  F1 (wgt)  : {test_res['f1']:.4f}")
print(f"  Precision : {precision_score(test_res['labels'], test_res['preds'], average='weighted'):.4f}")
print(f"  Recall    : {recall_score(test_res['labels'], test_res['preds'], average='weighted'):.4f}")
try:
    auc = roc_auc_score(test_res["labels"], test_res["probs"])
    print(f"  ROC-AUC   : {auc:.4f}")
except Exception:
    auc = None
print("=" * 55)

print("\n📋 Full Classification Report:")
print(classification_report(
    test_res["labels"], test_res["preds"],
    target_names=list(CFG["id2label"].values())
))

In [ ]:
# ── Confusion matrix + ROC curve ──────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle("ViT Evaluation — Test Set", fontsize=14, fontweight="bold")

# ── Confusion matrix ──────────────────────────────────────────────────────
cm = confusion_matrix(test_res["labels"], test_res["preds"])
class_names = [CFG["id2label"][i] for i in range(CFG["num_labels"])]

sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=class_names, yticklabels=class_names,
    ax=axes[0], linewidths=0.5
)
axes[0].set_title("Confusion Matrix")
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("Actual")

# ── ROC curve ─────────────────────────────────────────────────────────────
if auc is not None:
    fpr, tpr, _ = roc_curve(test_res["labels"], test_res["probs"])
    axes[1].plot(fpr, tpr, "b-", lw=2, label=f"ViT  (AUC = {auc:.3f})")
    axes[1].plot([0, 1], [0, 1], "k--", lw=1, label="Random")
    axes[1].set_xlabel("False Positive Rate")
    axes[1].set_ylabel("True Positive Rate")
    axes[1].set_title("ROC Curve")
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
else:
    axes[1].text(0.5, 0.5, "AUC not available", ha="center")

plt.tight_layout()
plt.savefig(f"{CFG['output_dir']}/vit_confusion_roc.png", dpi=150, bbox_inches="tight")
plt.show()

---
## Cell 10 — ViT vs CNN Performance Comparison

In [ ]:
# ── Comparison table ──────────────────────────────────────────────────────
# If Member 1 has provided CNN metrics, fill them in below.
# Default placeholder values are shown — replace with real numbers.

vit_acc  = test_res["acc"]
vit_f1   = test_res["f1"]
vit_prec = precision_score(test_res["labels"], test_res["preds"], average="weighted")
vit_rec  = recall_score(test_res["labels"], test_res["preds"], average="weighted")
vit_auc  = auc if auc else 0.0
vit_params = 86_000_000   # ViT-Base params

# ── Placeholder CNN numbers (replace with Member 1's actual results) ──────
CNN_ACCURACY  = 0.78   # <- replace
CNN_F1        = 0.76   # <- replace
CNN_PRECISION = 0.77   # <- replace
CNN_RECALL    = 0.76   # <- replace
CNN_AUC       = 0.81   # <- replace
CNN_PARAMS    = 25_000_000

comparison_df = pd.DataFrame({
    "Model"            : ["CNN (Member 1)", "ViT (Member 2)"],
    "Architecture"     : ["ResNet / Custom CNN", "ViT-Base-Patch16-224"],
    "Parameters"       : [f"{CNN_PARAMS:,}", f"{vit_params:,}"],
    "Accuracy"         : [CNN_ACCURACY, vit_acc],
    "F1 (Weighted)"    : [CNN_F1, vit_f1],
    "Precision"        : [CNN_PRECISION, vit_prec],
    "Recall"           : [CNN_RECALL, vit_rec],
    "ROC-AUC"          : [CNN_AUC, vit_auc],
})

print("📊 Model Comparison Table:")
print(comparison_df.to_string(index=False))

In [ ]:
# ── Bar chart comparison ───────────────────────────────────────────────────

metrics = ["Accuracy", "F1 (Weighted)", "Precision", "Recall", "ROC-AUC"]
cnn_vals = [CNN_ACCURACY, CNN_F1, CNN_PRECISION, CNN_RECALL, CNN_AUC]
vit_vals = [vit_acc, vit_f1, vit_prec, vit_rec, vit_auc]

x = np.arange(len(metrics))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 6))
rects1 = ax.bar(x - width/2, cnn_vals, width, label="CNN (Member 1)", color="#3498db", alpha=0.85, edgecolor="white")
rects2 = ax.bar(x + width/2, vit_vals, width, label="ViT (Member 2)", color="#e74c3c", alpha=0.85, edgecolor="white")

ax.set_xlabel("Metric")
ax.set_ylabel("Score")
ax.set_title("CNN vs ViT — Performance Comparison", fontsize=14, fontweight="bold")
ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.set_ylim(0, 1.1)
ax.legend()
ax.grid(axis="y", alpha=0.3)

for rect, val in zip(rects1, cnn_vals):
    ax.text(rect.get_x() + rect.get_width()/2, rect.get_height() + 0.01, f"{val:.3f}", ha="center", va="bottom", fontsize=9)
for rect, val in zip(rects2, vit_vals):
    ax.text(rect.get_x() + rect.get_width()/2, rect.get_height() + 0.01, f"{val:.3f}", ha="center", va="bottom", fontsize=9)

plt.tight_layout()
plt.savefig(f"{CFG['output_dir']}/vit_vs_cnn_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

---
## Cell 11 — Extract & Save Image Feature Vectors for Member 5 🤝

> **This is the key output for the Multimodal model.**  
> Member 5 will load `vit_image_features.npy` (shape `N × 768`) and align it with `vit_image_features_meta.csv` by `listing_url`.

In [ ]:
# ── ViT backbone: extract CLS token embeddings ────────────────────────────

# Load the ViT backbone (without classification head) for feature extraction
vit_backbone = ViTModel.from_pretrained(best_model_path).to(DEVICE)
vit_backbone.eval()


@torch.no_grad()
def extract_features(model, loader, device):
    """
    Extract CLS token embeddings from ViT backbone.
    Returns:
        features : np.ndarray  shape (N, 768)
        labels   : np.ndarray  shape (N,)
    """
    all_features, all_labels = [], []

    for batch in tqdm(loader, desc="Extracting features"):
        pixel_values = batch["pixel_values"].to(device)
        labels       = batch["labels"]

        outputs = model(pixel_values=pixel_values)
        # CLS token is the first token of last_hidden_state → shape (B, 768)
        cls_embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()

        all_features.append(cls_embeddings)
        all_labels.extend(labels.numpy())

    return np.vstack(all_features), np.array(all_labels)


# ── Build a FULL dataset loader (no augmentation, preserving order) ────────
# We extract features for ALL images so Member 5 can align with text features
full_ds     = MobileImageDataset(df_imgs, transform=val_aug, label2id=label2id)
full_loader = DataLoader(full_ds, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

print(f"🔍 Extracting ViT image features for all {len(df_imgs):,} images ...")
features, feat_labels = extract_features(vit_backbone, full_loader, DEVICE)

print(f"✅ Feature matrix shape : {features.shape}")
print(f"   dtype                : {features.dtype}")
print(f"   Labels shape         : {feat_labels.shape}")

In [ ]:
# ── Save features + metadata ───────────────────────────────────────────────

feat_path = f"{CFG['output_dir']}/vit_image_features.npy"
meta_path = f"{CFG['output_dir']}/vit_image_features_meta.csv"

# 1. Save the raw feature matrix
np.save(feat_path, features)
print(f"💾 Saved feature matrix → {feat_path}  ({features.nbytes / 1e6:.1f} MB)")

# 2. Save aligned metadata (index matches row in features.npy exactly)
meta_df = df_imgs[["listing_id", "listing_url", "image_url", "label", "label_int", "price_egp", "brand"]].copy().reset_index(drop=True)
meta_df["feat_index"]   = np.arange(len(meta_df))
meta_df["vit_pred_label"] = feat_labels   # 0=Fair, 1=Overpriced
meta_df.to_csv(meta_path, index=False, encoding="utf-8")
print(f"💾 Saved feature metadata → {meta_path}")

# ── Quick verification ────────────────────────────────────────────────────
loaded = np.load(feat_path)
print(f"\n🔍 Verification:")
print(f"   Loaded shape     : {loaded.shape}")
print(f"   Mean (first vec) : {loaded[0].mean():.6f}")
print(f"   Meta rows        : {len(meta_df)}")
print(f"   Meta columns     : {list(meta_df.columns)}")

In [ ]:
# ── Visualise feature space (t-SNE) ───────────────────────────────────────

from sklearn.manifold import TSNE

# Use a sample for speed
n_sample = min(1000, len(features))
idx_sample = np.random.choice(len(features), n_sample, replace=False)
feat_sample   = features[idx_sample]
label_sample  = feat_labels[idx_sample]

print(f"Running t-SNE on {n_sample} samples ...")
tsne = TSNE(n_components=2, perplexity=30, random_state=CFG["seed"], n_iter=1000)
emb  = tsne.fit_transform(feat_sample)

fig, ax = plt.subplots(figsize=(10, 8))
colors_map = {0: "#2ecc71", 1: "#e74c3c"}
for label_id, label_name in CFG["id2label"].items():
    mask = label_sample == label_id
    ax.scatter(emb[mask, 0], emb[mask, 1], c=colors_map[label_id],
               label=label_name, alpha=0.6, s=20, edgecolors="none")

ax.set_title("t-SNE of ViT Image Features (CLS Token)", fontsize=13, fontweight="bold")
ax.legend(markerscale=2)
ax.set_xlabel("t-SNE Dim 1")
ax.set_ylabel("t-SNE Dim 2")
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.savefig(f"{CFG['output_dir']}/vit_tsne_features.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"💾 Saved → {CFG['output_dir']}/vit_tsne_features.png")

---
## Cell 12 — Save All Metrics to JSON

In [ ]:
# ── Compile and save metrics ───────────────────────────────────────────────

metrics_dict = {
    "model"       : CFG["vit_model_name"],
    "dataset"     : {
        "total_images" : len(df_imgs),
        "train"        : len(df_train),
        "val"          : len(df_val),
        "test"         : len(df_test),
        "labels"       : df_imgs["label"].value_counts().to_dict(),
    },
    "training"    : {
        "epochs"       : CFG["epochs"],
        "batch_size"   : CFG["batch_size"],
        "lr"           : CFG["lr"],
        "weight_decay" : CFG["weight_decay"],
        "history"      : history,
    },
    "test_metrics": {
        "accuracy"     : float(round(test_res["acc"],   4)),
        "f1_weighted"  : float(round(test_res["f1"],    4)),
        "precision"    : float(round(precision_score(test_res["labels"], test_res["preds"], average="weighted"), 4)),
        "recall"       : float(round(recall_score(test_res["labels"], test_res["preds"], average="weighted"), 4)),
        "roc_auc"      : float(round(auc, 4)) if auc else None,
        "confusion_matrix": confusion_matrix(test_res["labels"], test_res["preds"]).tolist(),
        "classification_report": classification_report(
            test_res["labels"], test_res["preds"],
            target_names=list(CFG["id2label"].values()),
            output_dict=True
        ),
    },
    "feature_output": {
        "file"          : feat_path,
        "meta_file"     : meta_path,
        "shape"         : list(features.shape),
        "embedding_dim" : 768,
        "token"         : "CLS",
    },
    "comparison": {
        "CNN": {"accuracy": CNN_ACCURACY, "f1": CNN_F1, "auc": CNN_AUC},
        "ViT": {"accuracy": float(vit_acc), "f1": float(vit_f1), "auc": float(vit_auc)},
    }
}

metrics_path = f"{CFG['output_dir']}/vit_metrics.json"
with open(metrics_path, "w", encoding="utf-8") as f:
    json.dump(metrics_dict, f, indent=2, ensure_ascii=False)

print(f"💾 All metrics saved → {metrics_path}")
print(f"\n📋 Final Summary:")
print(f"   Test Accuracy  : {metrics_dict['test_metrics']['accuracy']}")
print(f"   Test F1        : {metrics_dict['test_metrics']['f1_weighted']}")
print(f"   Test ROC-AUC   : {metrics_dict['test_metrics']['roc_auc']}")
print(f"   Feature shape  : {features.shape}")

---
## Cell 13 — Output Files Summary

In [ ]:
# ── List all saved output files ────────────────────────────────────────────

print("📁 Member 2 — Output Files Summary")
print("=" * 65)

output_dir = Path(CFG["output_dir"])
for f in sorted(output_dir.rglob("*")):
    if f.is_file():
        size_mb = f.stat().st_size / 1e6
        print(f"  {str(f):<55} {size_mb:>7.2f} MB")

print("=" * 65)
print()
print("🤝 Files for Member 5 (Multimodal):")
print(f"   📦 vit_image_features.npy           — shape {features.shape}  (image embeddings)")
print(f"   📋 vit_image_features_meta.csv       — aligned metadata (listing_url, label, etc.)")
print(f"   🤖 vit_best_model/                   — full model weights for inference")
print()
print("📌 Usage by Member 5:")
print("""
    import numpy as np
    import pandas as pd

    # Load image features
    image_feats = np.load('outputs/vit_image_features.npy')  # (N, 768)
    meta        = pd.read_csv('outputs/vit_image_features_meta.csv')

    # Merge with text features by listing_url
    # (text_feats come from Member 3/4 — align on listing_url)
    combined = meta.merge(text_feature_df, on='listing_url')
    img_idx  = combined['feat_index'].values
    X_image  = image_feats[img_idx]   # aligned image features
""")

---
## Cell 14 — Quick Inference Demo

In [ ]:
# ── Demo: predict a single listing from URL ────────────────────────────────

@torch.no_grad()
def predict_from_url(image_url: str, model, device, transform=None):
    """
    Download an image by URL and return the predicted price label.
    Returns: dict with 'label', 'confidence', and class probabilities.
    """
    if transform is None:
        transform = val_aug

    # Download
    resp = requests.get(image_url, headers=HEADERS, timeout=10)
    img  = Image.open(resp.raw).convert("RGB")
    img_np = np.array(img)

    # Preprocess
    aug    = transform(image=img_np)
    tensor = aug["image"].unsqueeze(0).to(device)

    # Predict
    model.eval()
    output = model(pixel_values=tensor)
    probs  = torch.softmax(output.logits, dim=-1).squeeze().cpu().numpy()
    pred   = int(np.argmax(probs))

    return {
        "label"      : CFG["id2label"][pred],
        "confidence" : float(probs[pred]),
        "probs"      : {CFG["id2label"][i]: float(probs[i]) for i in range(len(probs))},
        "image"      : img,
    }


# ── Run on a sample from the dataset ─────────────────────────────────────
sample_row = df_imgs.sample(1, random_state=99).iloc[0]
result = predict_from_url(sample_row["image_url"], best_model, DEVICE)

fig, ax = plt.subplots(figsize=(6, 5))
ax.imshow(result["image"])
ax.set_title(
    f"🔮 Predicted: {result['label']}  ({result['confidence']*100:.1f}%)\n"
    f"   Actual: {sample_row['label']}  |  Price: EGP {sample_row['price_egp']:,.0f}",
    fontsize=11
)
ax.axis("off")
plt.tight_layout()
plt.show()

print(f"\nDescription: {sample_row['description']}")
print(f"Probabilities: {result['probs']}")

---
## ✅ Member 2 Deliverables Checklist

| # | Requirement | Status |
|---|-------------|--------|
| 1 | Fine-tune pre-trained ViT (`google/vit-base-patch16-224`) | ✅ Cell 7–8 |
| 2 | Image augmentation pipeline (Albumentations, 12+ transforms) | ✅ Cell 5 |
| 3 | Augmentation visualisation saved | ✅ `vit_augmented_samples.png` |
| 4 | Evaluation metrics: Accuracy, F1, Precision, Recall, ROC-AUC | ✅ Cell 9 |
| 5 | Confusion matrix + ROC curve plots | ✅ `vit_confusion_roc.png` |
| 6 | Compare ViT vs CNN performance | ✅ Cell 10 |
| 7 | Extract CLS token image features (768-dim) | ✅ Cell 11 |
| 8 | Save `vit_image_features.npy` for Member 5 | ✅ `outputs/` |
| 9 | Save aligned `vit_image_features_meta.csv` | ✅ `outputs/` |
| 10 | Save best model weights | ✅ `outputs/vit_best_model/` |
| 11 | Save all metrics to JSON | ✅ `vit_metrics.json` |
| 12 | t-SNE feature visualisation | ✅ `vit_tsne_features.png` |
| 13 | Demo inference from image URL | ✅ Cell 14 |